**Magdy's Model**

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X, y)
y_test_pred_full = rf.predict(X_test)
rf_importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

rf_importance_df

In [ ]:
print("Full RF Accuracy:", accuracy_score(y_test, y_test_pred_full))
print(classification_report(y_test, y_test_pred_full))

In [ ]:
tree_full = rf.estimators_[0]

plt.figure(figsize=(20,10))

plot_tree(
    tree_full,
    feature_names=X_train.columns,
    class_names=["No DR", "DR"],
    filled=True,
    max_depth=3,
    fontsize=10
)

plt.title("Sample Tree from Full Random Forest")
plt.show()

In [ ]:
top_n = 15

plt.figure(figsize=(10,6))
sns.barplot(data=rf_importance_df.head(top_n), x="Importance", y="Feature")
plt.title("Top Random Forest Feature Importances")
plt.show()

In [ ]:
top_rf_features = rf_importance_df.head(15)["Feature"].tolist()

strong_corr_features = corr_with_target[abs(corr_with_target) > 0.05].index.tolist()
strong_corr_features = [f for f in strong_corr_features if f in feature_cols]

final_selected_features = sorted(list(set(top_rf_features + strong_corr_features)))

print("Final selected features:")
print(final_selected_features)
print("\nCount:", len(final_selected_features))

In [ ]:
perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature": feature_cols,
    "Permutation Importance": perm.importances_mean
}).sort_values("Permutation Importance", ascending=False)

perm_df.head(20)

In [ ]:
top_n = 15

plt.figure(figsize=(10,6))
sns.barplot(
    data=perm_df.head(top_n),
    x="Permutation Importance",
    y="Feature"
)
plt.title("Top Permutation Importances")
plt.show()

In [ ]:
rf_top = set(rf_importance_df.head(15)["Feature"])
perm_top = set(perm_df.head(15)["Feature"])


final_selected_features = list((rf_top | perm_top) & (set(feature_cols)))
final_selected_features = [f for f in final_selected_features if f in feature_cols]

print("Selected features count:", len(final_selected_features))
print(final_selected_features)

In [ ]:
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_reduced.fit(X_train[final_selected_features], y_train)
y_test_pred_reduced = rf_reduced.predict(X_test[final_selected_features])

In [ ]:
print("Reduced RF Accuracy:", accuracy_score(y_test, y_test_pred_reduced))
print(classification_report(y_test, y_test_pred_reduced))

In [ ]:
tree_reduced = rf_reduced.estimators_[0]

plt.figure(figsize=(20,10))

plot_tree(
    tree_reduced,
    feature_names=final_selected_features,
    class_names=["No DR", "DR"],
    filled=True,
    max_depth=3,
    fontsize=10
)

plt.title("Sample Tree from Reduced Random Forest")
plt.show()c